<a href="https://colab.research.google.com/github/DiaaEddin963/ACE6313-Employee-Attrition-Group-I/blob/main/ACE6313_Group_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- PART A: DATA PREPROCESSING By Diaa
# 1. Data Cleaning
import pandas as pd
url = 'https://raw.githubusercontent.com/DiaaEddin963/ACE6313-Employee-Attrition-Group-I/main/WA_Fn-UseC_-HR-Employee-Attrition.csv'
df = pd.read_csv(url)

df = df.dropna()
df = df.drop_duplicates()
# Drop inconsistent or uninformative columns (e.g., standard hours is exactly 80 for everyone)
columns_to_drop = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
df_cleaned = df.drop(columns=columns_to_drop, errors='ignore')


In [ ]:
# 2. Data Transformation
from sklearn.preprocessing import StandardScaler

y = df_cleaned['Attrition'].map({'Yes': 1, 'No':0})
X = df_cleaned.drop('Attrition', axis=1)

X_encoded = pd.get_dummies(X, drop_first=True)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
X_transformed = pd.DataFrame(X_scaled, columns=X_encoded. columns)


In [ ]:
# 3. Data Reduction
from sklearn.decomposition import PCA

pca = PCA(n_components=0.90)
X_pca = pca.fit_transform(X_transformed)
print("PART A: DATA PREPROCESSING")
print("1. Data Cleaning Complete. Cleaned shape:", df_cleaned.shape)
print("2. Data Transformation Complete. Encoded shape:", X_transformed. shape)
print("3. Data Reduction Complete. PCA shape:", X_pca. shape)

PART A: DATA PREPROCESSING
1. Data Cleaning Complete. Cleaned shape: (1470, 31)
2. Data Transformation Complete. Encoded shape: (1470, 44)
3. Data Reduction Complete. PCA shape: (1470, 28)


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report
import pandas as pd

print("--- PREPARING DATA FOR MODELING ---")
# We use stratify=y to ensure the train and test sets have the same ratio of Yes/No attrition.
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

--- PREPARING DATA FOR MODELING ---
Training set size: 1176
Testing set size: 294


In [ ]:
from sklearn.linear_model import LogisticRegression

print("--- 1. LOGISTIC REGRESSION ---")
lr = LogisticRegression(class_weight='balanced',max_iter=1000, random_state=42)

# Define hyperparameters to tune
lr_params = {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["lbfgs", "liblinear"]
}

# Setup GridSearchCV optimizing for F1-score
lr_grid = GridSearchCV(lr, param_grid=lr_params, cv=5, scoring='f1', n_jobs=-1)
lr_grid.fit(X_train, y_train)

# Evaluate the best model
print(f"Best Hyperparameters: {lr_grid.best_params_}")
y_pred_lr = lr_grid.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))


--- 1. LOGISTIC REGRESSION ---
Best Hyperparameters: {'C': 0.1, 'solver': 'liblinear'}

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.79      0.86       247
           1       0.39      0.70      0.50        47

    accuracy                           0.78       294
   macro avg       0.66      0.75      0.68       294
weighted avg       0.85      0.78      0.80       294



In [ ]:
from sklearn.neighbors import KNeighborsClassifier

print("--- 5. K-NEAREST NEIGHBORS (kNN) ---")
knn = KNeighborsClassifier()

# Define hyperparameters to tune
knn_params = {
    "n_neighbors": [3, 5, 7, 9],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

# Setup GridSearchCV optimizing for F1-score
knn_grid = GridSearchCV(knn, param_grid=knn_params, cv=5, scoring='f1', n_jobs=-1)
knn_grid.fit(X_train, y_train)

# Evaluate the best model
print(f"Best Hyperparameters: {knn_grid.best_params_}")
y_pred_knn = knn_grid.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))


--- 5. K-NEAREST NEIGHBORS (kNN) ---
Best Hyperparameters: {'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'uniform'}

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.96      0.90       247
           1       0.36      0.11      0.16        47

    accuracy                           0.83       294
   macro avg       0.60      0.53      0.53       294
weighted avg       0.77      0.83      0.79       294

